In [22]:
!pip install sqlalchemy==1.3.9

Defaulting to user installation because normal site-packages is not writeable
  Using cached sqlalchemy-1.3.9-cp314-cp314-win_amd64.whl
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 2.0.50
    Uninstalling SQLAlchemy-2.0.50:
      Successfully uninstalled SQLAlchemy-2.0.50


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython-sql 0.5.0 requires sqlalchemy>=2.0, but you have sqlalchemy 1.3.9 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
!pip install ipython-sql
!pip install ipython-sql prettytable

Defaulting to user installation because normal site-packages is not writeable
  Using cached sqlalchemy-2.0.50-cp314-cp314-win_amd64.whl.metadata (9.8 kB)
Using cached sqlalchemy-2.0.50-cp314-cp314-win_amd64.whl (2.1 MB)
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 1.3.9
    Uninstalling SQLAlchemy-1.3.9:
      Successfully uninstalled SQLAlchemy-1.3.9



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [25]:
import csv, sqlite3
import prettytable
prettytable.DEFAULT = 'DEFAULT'

con = sqlite3.connect("my_data1.db")
cur = con.cursor()

In [26]:
!pip install -q pandas


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
%sql sqlite:///my_data1.db

In [28]:
import pandas as pd
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False,method="multi")

101

In [29]:
#DROP THE TABLE IF EXISTS

%sql DROP TABLE IF EXISTS SPACEXTABLE;

 * sqlite:///my_data1.db
Done.


[]

In [30]:
%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null

 * sqlite:///my_data1.db
Done.


[]

### Task 1
Display the names of the unique launch sites in the space mission.

In [31]:
%%sql SELECT DISTINCT Launch_Site
FROM SPACEXTABLE;

 * sqlite:///my_data1.db
Done.


Launch_Site
CCAFS LC-40
VAFB SLC-4E
KSC LC-39A
CCAFS SLC-40


### Task 2
Display 5 records where launch sites begin with the string 'CCA'.

In [32]:
%%sql SELECT *
FROM SPACEXTABLE
WHERE Launch_Site LIKE 'CCA%'
LIMIT 5;

 * sqlite:///my_data1.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### Task 3
Display the total payload mass carried by boosters launched by NASA (CRS).

In [33]:
%%sql SELECT SUM(PAYLOAD_MASS__KG_) AS TOTAL_PAYLOAD
FROM SPACEXTABLE
WHERE Customer = 'NASA (CRS)';

 * sqlite:///my_data1.db
Done.


TOTAL_PAYLOAD
45596


### Task 4
Display average payload mass carried by booster version F9 v1.1.

In [34]:
%%sql SELECT AVG(PAYLOAD_MASS__KG_) AS AVG_PAYLOAD
FROM SPACEXTABLE
WHERE Booster_Version = 'F9 v1.1';

 * sqlite:///my_data1.db
Done.


AVG_PAYLOAD
2928.4


### Task 5
List the date when the first successful landing outcome on a ground pad was achieved.

In [35]:
%%sql SELECT MIN(Date) AS FIRST_SUCCESS_GROUND_PAD
FROM SPACEXTABLE
WHERE Landing_Outcome = 'Success (ground pad)';

 * sqlite:///my_data1.db
Done.


FIRST_SUCCESS_GROUND_PAD
2015-12-22


### Task 6
List the names of boosters which have success in drone ship and have payload mass greater than 4000 but less than 6000.

In [36]:
%%sql SELECT Booster_Version
FROM SPACEXTABLE
WHERE Landing_Outcome = 'Success (drone ship)'
AND PAYLOAD_MASS__KG_ > 4000
AND PAYLOAD_MASS__KG_ < 6000;

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 FT B1022
F9 FT B1026
F9 FT B1021.2
F9 FT B1031.2


### Task 7
List the total number of successful and failure mission outcomes.

In [37]:
%%sql SELECT Mission_Outcome,
       COUNT(*) AS COUNT
FROM SPACEXTABLE
GROUP BY Mission_Outcome;

 * sqlite:///my_data1.db
Done.


Mission_Outcome,COUNT
Failure (in flight),1
Success,98
Success,1
Success (payload status unclear),1


### Task 8
List all booster versions that have carried the maximum payload mass.

In [38]:
%%sql SELECT Booster_Version
FROM SPACEXTABLE
WHERE PAYLOAD_MASS__KG_ =
(
    SELECT MAX(PAYLOAD_MASS__KG_)
    FROM SPACEXTABLE
);

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 B5 B1048.4
F9 B5 B1049.4
F9 B5 B1051.3
F9 B5 B1056.4
F9 B5 B1048.5
F9 B5 B1051.4
F9 B5 B1049.5
F9 B5 B1060.2
F9 B5 B1058.3
F9 B5 B1051.6


### Task 9
Display the month number, landing outcomes that failed on drone ship, booster versions, and launch sites for launches in 2015.

In [39]:
%%sql SELECT substr(Date,6,2) AS MONTH,
       Landing_Outcome,
       Booster_Version,
       Launch_Site
FROM SPACEXTABLE
WHERE substr(Date,1,4) = '2015'
AND Landing_Outcome = 'Failure (drone ship)';

 * sqlite:///my_data1.db
Done.


MONTH,Landing_Outcome,Booster_Version,Launch_Site
01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


### Task 10
Rank the count of landing outcomes between 2010-06-04 and 2017-03-20 in descending order.

In [40]:
%%sql SELECT Landing_Outcome,
       COUNT(*) AS COUNT
FROM SPACEXTABLE
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY COUNT DESC;

 * sqlite:///my_data1.db
Done.


Landing_Outcome,COUNT
No attempt,10
Success (drone ship),5
Failure (drone ship),5
Success (ground pad),3
Controlled (ocean),3
Uncontrolled (ocean),2
Failure (parachute),2
Precluded (drone ship),1
